In [69]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import string
import numpy as np
import pandas as pd
import seaborn as sns
from torch.utils.tensorboard import SummaryWriter

In [70]:
with open('girls-names.csv') as f:
    content = f.readlines()
print(len(content))
content = ' '.join( [c.strip()+'.' for c in content])
content[:100]

# allowed_chars = set(string.ascii_letters + string.digits + ". ")

# with open('persuasion.txt') as f:
#     content = f.read()
# print(len(content))
# content = ''.join([c for c in content if c in allowed_chars])
# content[:100]

3759


'Aadhya. Aadya. Aaira. Aairah. Aaliyah. Aanya. Aaradhya. Aarna. Aarvi. Aarya. Aashvi. Aavya. Aayat. A'

In [71]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(device)

mps


In [72]:
allowed_chars = set(string.ascii_letters + string.digits + ". ")

content = ''.join([c for c in content if c in allowed_chars])

chars = set(sorted(list(content)))
vocab_size = len(chars)
print(f'vocab_size is {vocab_size}')
i_to_c = {k:v for k, v in enumerate(chars)}
c_to_i = {k:v for v, k in i_to_c.items()}

encode = lambda s: [c_to_i[c] for c in s] # noqa: E731
decode = lambda s: ''.join([i_to_c[i] for i in s]) # noqa: E731
decode(encode(content[:100]))

encoded_content = torch.tensor(encode(content))
print(encoded_content.shape)
print(f'number of encoded chars {len(encoded_content):,}')
print(f'sample encoded content {encoded_content[:100]}')

vocab_size is 54
torch.Size([30060])
number of encoded chars 30,060
sample encoded content tensor([50, 16, 36, 12, 29, 16, 44, 25, 50, 16, 36, 29, 16, 44, 25, 50, 16, 11,
        17, 16, 44, 25, 50, 16, 11, 17, 16, 12, 44, 25, 50, 16,  5, 11, 29, 16,
        12, 44, 25, 50, 16, 19, 29, 16, 44, 25, 50, 16, 17, 16, 36, 12, 29, 16,
        44, 25, 50, 16, 17, 19, 16, 44, 25, 50, 16, 17, 21, 11, 44, 25, 50, 16,
        17, 29, 16, 44, 25, 50, 16,  1, 12, 21, 11, 44, 25, 50, 16, 21, 29, 16,
        44, 25, 50, 16, 29, 16, 14, 44, 25, 50])


In [73]:
context_length = 4

class TransformerNameGenerator(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd):
        super().__init__()
        
        self.token_embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)

        self.position_embedding = nn.Embedding(num_embeddings=context_length, embedding_dim=n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)
            

    def forward(self, idx):
        B, T = idx.shape
 
        token_embedding = self.token_embedding(idx) # B, T, embd_n
 
        pos = torch.arange(T)
 
        positional_embeddings = self.position_embedding(pos) # T, nemb
 
        x = token_embedding + positional_embeddings
 
        logits = self.lm_head(x)
 
        return logits

model = TransformerNameGenerator(len(chars), context_length, 512)


optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [74]:
from torch.utils.data import TensorDataset, DataLoader

xs = []
ys = []
for i in range(0, len(encoded_content)-context_length):
    x_chunk = encoded_content[i:i+context_length]
    y_chunk = encoded_content[i+1:i+context_length+1]

    xs.append(x_chunk)
    ys.append(y_chunk)

X = torch.stack(xs)
Y = torch.stack(ys)

dataset = TensorDataset(X, Y)

loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [75]:
for xb, yb in loader:
    logits = model.forward(xb)
    B, T, C = logits.shape
    logits = logits.view(B * T, C)
    targets = yb.view(B*T)
    loss = F.cross_entropy(logits, targets)
    print(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

4.353797435760498
3.3367416858673096
2.9031593799591064
3.748983144760132
3.9632132053375244
3.1479530334472656
3.4538440704345703
3.5407378673553467
3.0831665992736816
3.098522663116455
3.32147216796875
3.132023572921753
3.207698106765747
3.0210816860198975
3.3034842014312744
3.4274861812591553
3.1569573879241943
3.0861661434173584
2.8931450843811035
2.894763946533203
2.735905647277832
2.941016912460327
3.0290367603302
2.6303963661193848
3.1321732997894287
2.7532358169555664
3.577758550643921
3.1208817958831787
2.6818361282348633
2.473663806915283
3.030176877975464
2.543182611465454
2.6807451248168945
2.7740085124969482
3.0891189575195312
3.0485665798187256
3.2655861377716064
2.725285053253174
2.6898839473724365
2.8597664833068848
2.5918824672698975
2.93483567237854
2.6228816509246826
2.9916629791259766
2.823939800262451
3.2882795333862305
3.0713558197021484
2.707482099533081
3.1847341060638428
2.5702931880950928
2.930756092071533
2.817613124847412
2.8606317043304443
2.611837148666382

In [76]:
decode(model.forward(torch.tensor(encode('    ')).view(1,-1)).argmax(dim=2).tolist()[0])

'LAAL'